In [ ]:
from CodeSensor import *
from run import load_model, load_tokenizer
import pandas as pd

In [ ]:
df_cpp = pd.read_parquet('tests/cpp_tests_10k.parquet')
df_python = pd.read_parquet('tests/python_tests_10k.parquet')

In [ ]:
from torch.cuda import is_available
device = 'cuda:0' if is_available() else 'cpu'

tokenizer = load_tokenizer()
model_cpp = load_model('weights/cpp_old.pth', device)
model_py = load_model('weights/py_v2.pth', device)

sensor_cpp = CodeSensor(tokenizer, model_cpp, device)
sensor_py = CodeSensor(tokenizer, model_py, device)

### Пример тестирования модели для всего датасета
verdict = 0 - человек (доля подозрительного кода в пределах допустимого - <15%), \
verdict = 1 - содержит 15-30% подозрительного кода, \
verdict = 2 - содержит более 30% подозрительного кода

HG - Human generated \
MG - Machine generated \
MR - Machine refined\
MGA - AI-Generated-Adversarial Code

In [ ]:
df_cpp['verdict'] = df_cpp['code'].apply(lambda code : sensor_cpp.analyze(code).verdict)

df_cpp['fp_rate'] = df_cpp.apply(lambda row : int(row['verdict'] >= 1 and row['label'] == 'HG'), axis=1)
df_cpp['fn_rate'] = df_cpp.apply(lambda row : int(row['verdict'] < 1 and row['label'] != 'HG'), axis=1)

fp_rate = np.sum(df_cpp['fp_rate']) / len(df_cpp[df_cpp['label'] == 'HG'])
fn_rate = np.sum(df_cpp['fn_rate']) / len(df_cpp[df_cpp['label'] != 'HG'])
fp_rate, fn_rate